# 第 10 章：现代 LLaMA 架构

对应新版 `roadmap.md` 主线入口。

**核心问题**：主流开源大模型在教学版 GPT 上改了什么，为什么改？

**本章关注**：RoPE, RMSNorm, SwiGLU, GQA, KV cache。

**轻量实验**：比较 MHA 与 GQA 的 KV cache 元素数量，建立推理成本直觉。

> 说明：本 notebook 只做最小可运行观察，不做大型训练；如果后续要接入仓库内 API，请先确认 API 已存在。


In [ ]:
def kv_cache_values(batch, seq_len, num_kv_heads, head_dim):
    # K 和 V 各一份，所以乘以 2。
    return batch * seq_len * num_kv_heads * head_dim * 2

batch, seq_len, q_heads, kv_heads, head_dim = 1, 128, 8, 2, 16
mha_cache = kv_cache_values(batch, seq_len, q_heads, head_dim)
gqa_cache = kv_cache_values(batch, seq_len, kv_heads, head_dim)

print({
    "mha_cache_values": mha_cache,
    "gqa_cache_values": gqa_cache,
    "reduction_ratio": round(mha_cache / gqa_cache, 2),
})


## 学习观察

运行上面的最小实验后，建议记录三点：

1. 哪个输入或配置最影响输出？
2. 这个 toy 实验和本章核心问题之间的对应关系是什么？
3. 如果要进入 `src/` 或真实模型实现，还缺哪些已确认的 API、测试或数据？

本章验收时优先看能否解释：主流开源大模型在教学版 GPT 上改了什么，为什么改？
